In [30]:
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score
from sklearn.preprocessing import label_binarize
from torch.mps import empty_cache
import gc

In [2]:
hyper_params_KNN = {
    'n_neighbors': [3, 5, 7],
}

hyper_params_RF = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10],
}

hyper_params_SVC = {
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale']
}

In [3]:
def best_model(model, hyper_params, X, y):
    gscv = GridSearchCV(
        model,
        hyper_params,
        cv=2,
        n_jobs=1
    )
    gscv.fit(X, y)
    return gscv.best_estimator_

In [44]:
def evaluate(y_true, y_pred, y_proba=None):
    results = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average='weighted'),
        "recall": recall_score(y_true, y_pred, average='weighted'),
        "f1": f1_score(y_true, y_pred, average='weighted'),
        "mcc": matthews_corrcoef(y_true, y_pred)
    }

    if y_proba is not None:
        if len(np.unique(y_true)) == 2:
            results["auc"] = roc_auc_score(y_true, y_proba[:, 1])
        else:
            results["auc"] = roc_auc_score(
                y_true,
                y_proba,
                multi_class='ovr',
                average='weighted'
            )
    return results

In [5]:
def manual_cv(model, X, y, skf, need_proba=False):
    y_pred = np.zeros_like(y)

    if need_proba:
        y_proba = np.zeros((len(y), len(np.unique(y))))
    else:
        y_proba = None

    for train_idx, test_idx in skf.split(X, y):
        model.fit(X[train_idx], y[train_idx])
        y_pred[test_idx] = model.predict(X[test_idx])

        if need_proba:
            y_proba[test_idx] = model.predict_proba(X[test_idx])

    return y_pred, y_proba

In [58]:
CA_data = np.load("Custard Apple_data.npz")
X = CA_data["X"]
y = CA_data["y"]

skf = StratifiedKFold(n_splits=5)

knn = best_model(KNeighborsClassifier(), hyper_params_KNN, X, y)
rf = best_model(RandomForestClassifier(), hyper_params_RF, X, y)
svc = best_model(SVC(), hyper_params_SVC, X, y)

y_pred_knn, y_proba_knn = manual_cv(knn, X, y, skf, need_proba=True)
y_pred_rf, y_proba_rf = manual_cv(rf, X, y, skf, need_proba=True)
y_pred_svc, _ = manual_cv(svc, X, y, skf, need_proba=False)

print("KNN:", evaluate(y, y_pred_knn, y_proba_knn))
print("RF:", evaluate(y, y_pred_rf, y_proba_rf))
print("SVM:", evaluate(y, y_pred_svc))

del X, y, knn, rf, svc, y_pred_knn, y_proba_knn, y_pred_rf, y_proba_rf, y_pred_svc
gc.collect()
empty_cache()

KNN: {'accuracy': 0.9890577507598785, 'precision': 0.9890937977167581, 'recall': 0.9890577507598785, 'f1': 0.9890446004432786, 'mcc': 0.9867251519791356, 'auc': 0.997031721718058}
RF: {'accuracy': 0.9781155015197568, 'precision': 0.9781626148044044, 'recall': 0.9781155015197568, 'f1': 0.9780252101297752, 'mcc': 0.9734542128318256, 'auc': 0.9992424615632}
SVM: {'accuracy': 0.9933130699088146, 'precision': 0.9933118461314506, 'recall': 0.9933130699088146, 'f1': 0.9933055920800863, 'mcc': 0.9918846726489846}


In [50]:
FL_data = np.load("Fig Leaf_data.npz")
X = FL_data["X"]
y = FL_data["y"]

knn = best_model(KNeighborsClassifier(), hyper_params_KNN, X, y)
rf = best_model(RandomForestClassifier(), hyper_params_RF, X, y)
svc = best_model(SVC(), hyper_params_SVC, X, y)

y_pred_knn, y_proba_knn = manual_cv(knn, X, y, skf, need_proba=True)
y_pred_rf, y_proba_rf = manual_cv(rf, X, y, skf, need_proba=True)
y_pred_svc, _ = manual_cv(svc, X, y, skf, need_proba=False)

print("KNN:", evaluate(y, y_pred_knn, y_proba_knn))
print("RF:", evaluate(y, y_pred_rf, y_proba_rf))
print("SVM:", evaluate(y, y_pred_svc))

del X, y, knn, rf, svc, y_pred_knn, y_proba_knn, y_pred_rf, y_proba_rf, y_pred_svc
gc.collect()
empty_cache()

KNN: {'accuracy': 0.896551724137931, 'precision': 0.8965022904285354, 'recall': 0.896551724137931, 'f1': 0.8958933356453697, 'mcc': 0.7825764639521541, 'auc': 0.9454483695652175}
RF: {'accuracy': 0.896551724137931, 'precision': 0.896353547364249, 'recall': 0.896551724137931, 'f1': 0.8960143568321691, 'mcc': 0.782627098932459, 'auc': 0.9658482142857142}
SVM: {'accuracy': 0.8987068965517241, 'precision': 0.8984802359077505, 'recall': 0.8987068965517241, 'f1': 0.8982382936528369, 'mcc': 0.7872221263514447}


In [54]:
PL_data = np.load("Potato Leaf_data.npz")
X = PL_data["X"]
y = PL_data["y"]

knn = best_model(KNeighborsClassifier(), hyper_params_KNN, X, y)
rf = best_model(RandomForestClassifier(), hyper_params_RF, X, y)
svc = best_model(SVC(), hyper_params_SVC, X, y)

y_pred_knn, y_proba_knn = manual_cv(knn, X, y, skf, need_proba=True)
y_pred_rf, y_proba_rf = manual_cv(rf, X, y, skf, need_proba=True)
y_pred_svc, _ = manual_cv(svc, X, y, skf, need_proba=False)

print("KNN:", evaluate(y, y_pred_knn, y_proba_knn))
print("RF:", evaluate(y, y_pred_rf, y_proba_rf))
print("SVM:", evaluate(y, y_pred_svc))

del X, y, knn, rf, svc, y_pred_knn, y_proba_knn, y_pred_rf, y_proba_rf, y_pred_svc
gc.collect()
empty_cache()

KNN: {'accuracy': 0.6487804878048781, 'precision': 0.6584186924928481, 'recall': 0.6487804878048781, 'f1': 0.6406699571410389, 'mcc': 0.5743952749225011, 'auc': 0.8777170884122618}
RF: {'accuracy': 0.6715447154471544, 'precision': 0.6606845052803981, 'recall': 0.6715447154471544, 'f1': 0.658343748464356, 'mcc': 0.5973868852201281, 'auc': 0.9152223277851526}
SVM: {'accuracy': 0.7252032520325203, 'precision': 0.724950511772055, 'recall': 0.7252032520325203, 'f1': 0.724502769755223, 'mcc': 0.6655675883425688}


/Users/poo.ping/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
